# W5D4 — Build a Transformer Block — Lab

**Week 5 · Day 4 · NLP Foundations** · Lab

Assembly, not invention. `attention` was written yesterday, tested against the slide, and is sitting
on disk in `attention_check.json`. Everything added today is either three lines long or something
you have already built.

What is new is one addition, in the literal sense: a vector that depends on **where** a token is,
added to the token before it is scored. Yesterday's proof — reverse the values, get the identical
output — stops holding the moment you do that, and the warm-up watches it stop.

The graded skill today is not the transformer. It is **shape discipline**: an assert after every
single step, recorded in `block_shapes.json`. That sounds like bookkeeping until you meet the bug
this prevents. A wrong shape inside a transformer block usually does not raise — it broadcasts,
silently, and hands you a model that trains to a perfectly plausible loss curve and predicts
nothing. It will cost you a week 6 afternoon if you have not built the habit here.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٤ — بناء كتلة محوّل

**الأسبوع الخامس · اليوم الرابع · أساسيات معالجة اللغة** · معمل

تركيبٌ لا اختراع. فـ`attention` كُتبت أمس وفُحصت مقابل الشريحة وهي على القرص في
`attention_check.json`. وكل ما يُضاف اليوم إمّا في ثلاثة أسطر وإمّا شيء بنيته أصلًا.

والجديد إضافةٌ واحدة بالمعنى الحرفي: متّجهٌ يتوقّف على **موضع** الرمز، يُضاف إلى الرمز قبل تقييمه.
وبرهان الأمس — اعكس القيم فتحصل على الخرج المتطابق — يتوقّف عن الصحّة لحظة فعلك ذلك، ويراقب الإحماء
توقّفه.

والمهارة المُقيَّمة اليوم ليست المحوّل بل **انضباط الأشكال**: فحصٌ بعد كل خطوة، مُسجَّل في
`block_shapes.json`. ويبدو ذلك مسكًا للدفاتر حتى تلقى الخلل الذي يمنعه. فالشكل الخاطئ داخل كتلة
محوّل لا يرفع استثناءً عادةً — بل يُبثّ صامتًا، فيُسلّمك نموذجًا يتدرّب إلى منحنى خسارةٍ معقول تمامًا
ولا يتنبّأ بشيء. وسيُكلّفك ظهيرةً في الأسبوع السادس إن لم تبنِ العادة هنا.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Show that adding a position vector makes attention order-sensitive, with the numbers to prove it.
- Implement sinusoidal positional encoding and say why it is a function rather than a parameter.
- Split `d_model` across heads, run attention per head, concatenate and project.
- Raise a readable error for the one shape mistake every student makes.
- Assemble the full block — attention, residual, `LayerNorm`, feed-forward, residual, `LayerNorm` —
  and prove it is shape-preserving.
- Match your hand-built multi-head attention against `nn.MultiheadAttention` to 1e-5.
- Add a causal mask and say what training without one produces.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُظهر أن إضافة متّجه موضع تجعل الانتباه حسّاسًا للترتيب، بالأرقام التي تُبرهن ذلك.
- أن تُنفّذ ترميز الموضع الجَيبي وأن تقول لماذا هو دالّة لا معامل.
- أن تقسم `d_model` على الرؤوس، وتُشغّل الانتباه لكل رأس، وتضمّ وتُسقِط.
- أن ترفع خطأً مقروءًا لخطأ الشكل الوحيد الذي يقع فيه كل طالب.
- أن تُركّب الكتلة كاملةً — انتباه وبقيّة و`LayerNorm` وتغذية أمامية وبقيّة و`LayerNorm` — وأن
  تُبرهن أنها حافظة للشكل.
- أن تُطابق انتباهك متعدّد الرؤوس المبنيّ يدويًا مع `nn.MultiheadAttention` إلى ١e−٥.
- أن تضيف قناعًا سببيًا وأن تقول ما يُنتجه التدريب بدونه.

</div>


## About the data

**Almost none, and that is the point.** The block is built and tested on the three hard-coded
vectors from Wednesday and on random tensors of known shape, because a shape bug is easiest to see
when you know what the shape should be.

`reviews_sentiment` appears once, at the end of task 2.6, for a single sanity run: one real review,
tokenised on whitespace against a vocabulary built from the corpus, embedded with `nn.Embedding`,
and pushed through the finished block. There is no training and no label — the question is only
whether a real variable-length token sequence survives the thing you built.

**No downloads.** The tokeniser here is a regular expression and a dictionary, deliberately: a
`from_pretrained` call would add a network dependency to a lab that has no need of one, and Friday
introduces real subword tokenisation with the full explanation it deserves.

<div dir="rtl" align="right">

## عن البيانات

**تكاد لا توجد، وهذا هو المقصد.** فالكتلة تُبنى وتُفحص على المتّجهات الثلاثة المكتوبة بثبوت من
الأربعاء وعلى مُوتِّرات عشوائية بأشكال معلومة، لأن خلل الشكل أسهل ما يُرى حين تعرف ما يجب أن يكون
الشكل.

وتظهر `reviews_sentiment` مرّةً واحدة في نهاية المهمة ٢٫٦ لتشغيلة سلامةٍ واحدة: مراجعة حقيقية
واحدة، مُقسَّمة على المسافات مقابل معجمٍ مبنيّ من المُدوّنة، ومُمثَّلة بـ`nn.Embedding`، ومدفوعة عبر
الكتلة المنتهية. ولا تدريب ولا تسمية — والسؤال فقط هل تنجو متتالية رموزٍ حقيقية متغيّرة الطول من
الشيء الذي بنيته.

**ولا تنزيل.** فالمُقسِّم هنا تعبير نمطي وقاموس، عن قصد: فنداء `from_pretrained` يضيف تبعية شبكة إلى
معملٍ لا يحتاجها، ويوم الجمعة يُدخل التقسيم الجزئي الحقيقي بالشرح الذي يستحقّه.

</div>


## Setup

Everything is PyTorch on the CPU, on tensors small enough to print. `torch.manual_seed(42)` comes
from `seed_everything`, so the random projections are the same on every machine — which matters,
because task 2.6 compares your numbers against PyTorch's own to five decimal places.

<div dir="rtl" align="right">

## الإعداد

كل شيء PyTorch على المعالج، على مُوتِّرات صغيرة بما يكفي لطباعتها. و`torch.manual_seed(42)` يأتي من
`seed_everything`، فتكون الإسقاطات العشوائية نفسها على كل جهاز — وهذا مهمّ، لأن المهمة ٢٫٦ تقارن
أرقامك بأرقام PyTorch نفسها إلى خمس منازل عشرية.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("matplotlib")
seed_everything(42)

import json
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

use_course_style()
torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)

SEED = 42
D_MODEL = 8          # small enough to print, divisible by 2 and by 4
N_HEADS = 2
D_FF = 32            # the feed-forward's inner width, 4x d_model as in the paper
SEQ_LEN = 5

# Wednesday's three tokens and the query, and this morning's positional vectors.
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
QUERY = np.array([1.0, 0.0])
POSITIONS = np.array([[0.1, 0.0],
                      [0.0, 0.1],
                      [0.1, 0.1]])

# The helper signatures for task 2.4 — you write the bodies, not the plumbing.
def split_heads(x, n_heads):
    """(batch, seq, d_model) -> (batch, n_heads, seq, d_head)."""
    batch, seq, d_model = x.shape
    return x.view(batch, seq, n_heads, d_model // n_heads).transpose(1, 2)


def merge_heads(x):
    """(batch, n_heads, seq, d_head) -> (batch, seq, d_model)."""
    batch, n_heads, seq, d_head = x.shape
    return x.transpose(1, 2).contiguous().view(batch, seq, n_heads * d_head)


yesterday = json.loads(load_artefact("attention_check.json").read_text(encoding="utf-8"))
print(f"yesterday's output: {yesterday['output']} | order_invariant: "
      f"{yesterday['order_invariant']}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: one addition, and order appears  (≈25 min)

Everything here works, and it is the whole reason today exists.

First, reload yesterday's artefact and re-run yesterday's function on the same three vectors.
`[0.84, 0.58]`, unchanged — a regression test on your own work from twenty-four hours ago.

Then add this morning's positional vectors — `[0.1, 0]`, `[0, 0.1]`, `[0.1, 0.1]` — to the three
tokens and recompute. The weights move from `[0.42, 0.16, 0.42]` to something else, because each
token's score now depends on where it sits.

And then the part that matters — including a wrinkle worth more than the tidy version.

Reorder the tokens with the positions left in place, and the output **changes**. One addition, and
the mechanism can finally tell `the dog bit the man` from `the man bit the dog`.

But try it as a **full reversal** first and you get the identical output *again*, positions and all.
That is not a bug in the positions: with these three particular vectors and this particular query,
reversing them leaves the three scores a permutation of themselves, and a sum does not care.
Swapping the first two tokens breaks it immediately — the weights go from `[0.43, 0.14, 0.43]` to
something with no symmetry left in it.

The lesson is not about positional encoding. It is that a single test case passing proves one test
case. The order-sensitivity is real, and one badly chosen permutation would have let you conclude
the opposite.

Change a positional vector and watch how much of the output moves.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: إضافة واحدة فيظهر الترتيب (نحو ٢٥ دقيقة)

كل ما هنا يعمل، وهو سبب وجود اليوم كله.

أولًا، أعِد تحميل أثر الأمس وأعِد تشغيل دالّة الأمس على المتّجهات الثلاثة نفسها. `[0.84, 0.58]` بلا
تغيير — وهذا فحص انحدارٍ على عملك أنت قبل أربعٍ وعشرين ساعة.

ثم أضف متّجهات الموضع من هذا الصباح — `[0.1, 0]` و`[0, 0.1]` و`[0.1, 0.1]` — إلى الرموز الثلاثة
وأعِد الحساب. فتتحرّك الأوزان من `[0.42, 0.16, 0.42]` إلى غيرها، لأن درجة كل رمز صارت تتوقّف على
موضعه.

ثم الجزء المهمّ. فعكس ترتيب الرموز أمس أبقى الخرج **متطابقًا بتًّا ببت**. افعله الآن مرّةً أخرى
والمواضع مُلحَقة: تبقى متّجهات الموضع في أماكنها وتتحرّك الرموز عبرها، فيتغيّر الخرج. إضافةٌ واحدة،
فتستطيع الآليّة أخيرًا أن تميّز `the dog bit the man` من `the man bit the dog`.

غيّر متّجه موضعٍ وراقب كم من الخرج يتحرّك.

</div>


In [ ]:
def softmax(scores, axis=-1):
    """Yesterday's numerically stable softmax."""
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)


def attention(q, values):
    """Yesterday's single-query attention. Returns (output, weights)."""
    weights = softmax(values @ q)
    return weights @ values, weights


output_plain, weights_plain = attention(QUERY, V)
print(f"yesterday, reproduced : weights {weights_plain.round(2)} -> output "
      f"{output_plain.round(2)}   (artefact: {yesterday['output']})")

output_positioned, weights_positioned = attention(QUERY, V + POSITIONS)
print(f"with positions added  : weights {weights_positioned.round(2)} -> output "
      f"{output_positioned.round(2)}")

# Reorder the tokens, but leave the positions where they are — position 0 is still position 0.
PERMUTATIONS = {"reversed": [2, 1, 0], "first two swapped": [1, 0, 2]}

print(f"\nsame three tokens, reordered, positions unchanged:\n")
ORDER_INVISIBLE_WITHOUT, ORDER_VISIBLE_WITH = True, False
for name, order in PERMUTATIONS.items():
    reordered = V[order]
    plain_again, _ = attention(QUERY, reordered)
    positioned_again, weights_again = attention(QUERY, reordered + POSITIONS)

    invisible = np.array_equal(plain_again, output_plain)
    visible = not np.allclose(positioned_again, output_positioned, atol=1e-9)
    ORDER_INVISIBLE_WITHOUT &= invisible
    ORDER_VISIBLE_WITH |= visible

    print(f"  {name}")
    print(f"    without positions: {plain_again.round(4)}  identical to the original: {invisible}")
    print(f"    with positions   : {positioned_again.round(4)}  changed: {visible}"
          f"   weights {weights_again.round(2)}")

print(f"\norder invisible without positions, under every permutation: "
      f"{ORDER_INVISIBLE_WITHOUT}")
print(f"order visible with positions, under at least one            : {ORDER_VISIBLE_WITH}")
print(f"\nthe full reversal is the interesting one: it leaves the three scores a permutation of "
      f"themselves, so the sum is unchanged even with the positions added. One test case passing "
      f"proves one test case")

## Section 2 — Core: six tasks  (≈60 min)

1. Sinusoidal positional encoding, plotted, and two things checked about it.
2. Q, K and V as three `nn.Linear` projections, matched by hand.
3. The `√d` divisor, and what softmax does with and without it.
4. Multi-head attention from scratch, and a readable error for the one mistake everybody makes.
5. The complete block, with an assert after every step, written to `block_shapes.json`.
6. Your multi-head against `nn.MultiheadAttention` to 1e-5, then a real review through the block.

**Watch the clock on task 4.** The split and merge helpers are already in the setup cell — you
write the attention, not the tensor plumbing.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. ترميز الموضع الجَيبي، مرسومًا، وشيئان مفحوصان فيه.
٢. Q وK وV كثلاثة إسقاطات `nn.Linear`، مُطابَقة يدويًا.
٣. مقسوم `√d`، وما تفعله softmax به وبدونه.
٤. الانتباه متعدّد الرؤوس من الصفر، وخطأٌ مقروء للغلطة الوحيدة التي يقع فيها الجميع.
٥. الكتلة كاملةً، بفحصٍ بعد كل خطوة، مكتوبًا في `block_shapes.json`.
٦. انتباهك متعدّد الرؤوس مقابل `nn.MultiheadAttention` إلى ١e−٥، ثم مراجعة حقيقية عبر الكتلة.

**وراقب الساعة في المهمة الرابعة.** فمساعدتا القسمة والضمّ في خليّة الإعداد أصلًا — وأنت تكتب
الانتباه لا سباكة المُوتِّرات.

</div>


### Task 2.1 — sinusoidal positional encoding

The warm-up's three positional vectors were made up. The real thing is a formula: for position `pos`
and dimension `i`, a sine or a cosine of `pos / 10000^(2i/d_model)`, alternating between the two
across dimensions.

Implement it, then plot the `(seq_len, d_model)` matrix as a heatmap. You will see stripes: low
dimensions oscillate quickly with position, high dimensions barely change at all — so a token's
vector encodes its position at several scales at once.

Then check two things, because they are the two properties the design depends on:

1. **Two different positions get different vectors.** Obvious, and worth asserting, because an
   off-by-one in the exponent silently gives every position the same encoding.
2. **Position 5 is identical across two separate calls.** It is a *function* of the position, not a
   learned parameter — nothing to train, and a sequence longer than anything in training still gets
   a valid encoding.

<div dir="rtl" align="right">

### المهمة ٢٫١ — ترميز الموضع الجَيبي

متّجهات الموضع الثلاثة في الإحماء كانت مُختلقة. أما الحقيقي فصيغة: للموضع `pos` والبُعد `i`، جَيبٌ أو
جَيب تمامٍ لـ`pos / 10000^(2i/d_model)`، متبادلين بينهما على الأبعاد.

نفّذها، ثم ارسم مصفوفة `(seq_len, d_model)` خريطةً حرارية. فترى خطوطًا: الأبعاد الدنيا تتذبذب سريعًا
مع الموضع، والعليا تكاد لا تتغيّر — فيُرمّز متّجه الرمز موضعه بمقاييس عدّة في وقتٍ واحد.

ثم افحص شيئين، فهما الخاصّيتان اللتان يتوقّف عليهما التصميم:

١. **موضعان مختلفان يأخذان متّجهين مختلفين.** بديهي، ويستحقّ الفحص، لأن خطأً بواحدٍ في الأُسّ يُعطي
   كل موضعٍ الترميز نفسه صامتًا.
٢. **الموضع الخامس متطابق في نداءين منفصلين.** فهو **دالّة** للموضع لا معاملًا مُتعلَّمًا — فلا شيء
   يُدرَّب، والمتتالية الأطول من كل ما في التدريب تأخذ ترميزًا صحيحًا على أي حال.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build a (seq_len, 1) column of positions and a (d_model/2,) row of frequencies,
#    then broadcast: angles = position / (10000 ** (2i / d_model)).
# 2) Even dimensions take the sine of the angles, odd dimensions the cosine. Slicing with
#    [:, 0::2] and [:, 1::2] assigns both halves without a loop.
# 3) Plot with plt.imshow and label the axes position and dimension.
# 4) Then compare two rows, and call the function twice and compare row 5 with itself.
# Search: "sinusoidal positional encoding implementation torch"
# https://pytorch.org/docs/stable/generated/torch.arange.html
#
# ١) ابنِ عمودًا `(seq_len, 1)` من المواضع وصفًّا `(d_model/2,)` من الترددات، ثم بُثّهما:
#    `angles = position / (10000 ** (2i / d_model))`.
# ٢) الأبعاد الزوجية تأخذ جَيب الزوايا، والفردية جَيب تمامها. والاقتطاع بـ`[:, 0::2]`
#    و`[:, 1::2]` يُسنِد النصفين بلا حلقة.
# ٣) ارسم بـ`plt.imshow` وسمِّ المحورين بالموضع والبُعد.
# ٤) ثم قارن صفّين، ونادِ الدالّة مرّتين وقارن الصف الخامس بنفسه.
# ابحث عن: "sinusoidal positional encoding implementation torch"
# https://pytorch.org/docs/stable/generated/torch.arange.html
# ────────────────────────────────────────────────────────────────────

    # TODO: Build the angle matrix, then fill even dimensions with sin and odd with cos.
    # مهمة: ابنِ مصفوفة الزوايا، ثم املأ الأبعاد الزوجية بـ`sin` والفردية بـ`cos`.
# TODO: different positions differ, and the same position is stable across calls.
# مهمة: تختلف، والموضع نفسه مستقرّ بين النداءات.

### Task 2.2 — Q, K and V are three projections

In self-attention every token produces three vectors from itself: a query (what am I looking for), a
key (what do I offer) and a value (what I pass on). Each is a linear projection — an `nn.Linear` —
and the three sets of weights are what the model learns.

Build the three projections with `d_model` in and `d_model` out, then do the thing that turns this
from a library call into understanding: **take one token, project it, and reproduce the same numbers
by hand** with `x @ W.T + b`. `nn.Linear` stores its weight transposed, and that transpose is the
most common source of a silently wrong reimplementation.

Print the maximum difference between your arithmetic and the module's output. It should be 0.0, not
"small".

<div dir="rtl" align="right">

### المهمة ٢٫٢ — Q وK وV ثلاثة إسقاطات

في الانتباه الذاتي يُنتج كل رمزٍ ثلاثة متّجهات من نفسه: مُستعلِمًا (عمّا أبحث) ومفتاحًا (ما أعرض)
وقيمةً (ما أُمرّر). وكلٌّ منها إسقاط خطّي — `nn.Linear` — ومجموعات الأوزان الثلاث هي ما يتعلّمه
النموذج.

ابنِ الإسقاطات الثلاثة بـ`d_model` دخلًا و`d_model` خرجًا، ثم افعل الشيء الذي يحوّل هذا من نداء
مكتبةٍ إلى فهم: **خُذ رمزًا واحدًا، وأسقِطه، وأعِد إنتاج الأعداد نفسها يدويًا** بـ`x @ W.T + b`.
فـ`nn.Linear` تحفظ وزنها منقولًا، وذلك النقل أشيع مصادر إعادة تنفيذٍ خاطئةٍ صامتة.

اطبع أكبر فرق بين حسابك وخرج الوحدة. ويجب أن يكون ٠٫٠ لا «صغيرًا».

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Three nn.Linear(D_MODEL, D_MODEL) modules. Keep them in a dict or as three names.
# 2) X is a (1, SEQ_LEN, D_MODEL) tensor — build it once from torch.randn and reuse it
#    for the rest of the lab.
# 3) By hand: x @ layer.weight.T + layer.bias. If you use layer.weight without the .T
#    the shapes still work when in and out are equal — which is exactly why this is a
#    silent bug rather than a crash.
# 4) Compare with torch.allclose and print the max absolute difference.
# Search: "pytorch nn.Linear weight shape transpose manual matmul"
# https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
#
# ١) ثلاث وحدات `nn.Linear(D_MODEL, D_MODEL)`. احفظها في قاموس أو بثلاثة أسماء.
# ٢) و`X` مُوتِّر `(1, SEQ_LEN, D_MODEL)` — ابنِه مرّةً من `torch.randn` وأعِد استخدامه
#    في بقيّة المعمل.
# ٣) ويدويًا: `x @ layer.weight.T + layer.bias`. فإن استخدمت `layer.weight` بلا `.T`
#    فالأشكال تتحاذى حين يتساوى الدخل والخرج — وهذا بعينه سبب كونه خللًا صامتًا لا
#    انهيارًا.
# ٤) قارن بـ`torch.allclose` واطبع أكبر فرق مطلق.
# ابحث عن: "pytorch nn.Linear weight shape transpose manual matmul"
# https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
# ────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
X = torch.randn(1, SEQ_LEN, D_MODEL)
# TODO: with the transposed weight and the bias. Print the maximum difference.
# مهمة: المنقول والانحياز. واطبع أكبر فرق.

### Task 2.3 — the `√d` divisor

Scaled dot-product attention divides the scores by `√d_head` before the softmax. Compute one score
both ways and compare.

The unscaled score is larger by a factor of `√d`. On its own that is uninteresting; what matters is
what softmax does with it. Print both softmax distributions and their maximum weight.

The reason is a variance argument, and it is worth being able to state: a dot product of `d`
roughly-independent terms has a standard deviation around `√d`, so as `d` grows the scores spread
out, and softmax of a widely spread vector approaches one-hot. A one-hot attention distribution has
almost no gradient — the model stops learning for a reason that nothing in the loss curve explains.
You measured exactly this at the end of yesterday's stretch; this is the same fact with the fix
applied.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — مقسوم `√d`

يقسم الانتباه المقيس بالجداء القياسي الدرجات على `√d_head` قبل softmax. احسب درجةً واحدة
بالطريقتين وقارن.

فالدرجة غير المقسومة أكبر بمعامل `√d`. وهذا وحده غير مثير؛ والمهمّ ما تفعله softmax به. اطبع
توزيعَي softmax وأكبر وزنٍ في كلٍّ منهما.

والسبب حجّة تباين، ويستحقّ أن تقدر على قولها: فجداءٌ قياسي من `d` حدًّا مستقلًّا تقريبًا انحرافه
المعياري نحو `√d`، فبنموّ `d` تتباعد الدرجات، وsoftmax لمتّجهٍ واسع التباعد يقارب الأحادي الساخن.
وتوزيع انتباهٍ أحاديّ ساخن يكاد لا يكون له تدرّج — فيتوقّف النموذج عن التعلّم لسببٍ لا يشرحه شيء في
منحنى الخسارة. وقد قِست هذا بعينه في نهاية تمديد الأمس؛ وهذه الحقيقة نفسها والإصلاح مُطبَّق.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) d_head is D_MODEL // N_HEADS. The divisor is its square root, not d_model's.
# 2) Take Q[0, 0] and K[0] from task 2.2 and compute the scores for one query against
#    all SEQ_LEN keys, both with and without the division.
# 3) Softmax both and print the two weight vectors and their maxima.
# Search: "scaled dot product attention sqrt d_k variance"
# https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
#
# ١) و`d_head` هو `D_MODEL // N_HEADS`. والمقسوم جذره التربيعي لا جذر `d_model`.
# ٢) خُذ `Q[0, 0]` و`K[0]` من المهمة ٢٫٢ واحسب درجات مُستعلِمٍ واحد مقابل المفاتيح
#    `SEQ_LEN` كلها، بالقسمة وبدونها.
# ٣) طبّق softmax على الاثنين واطبع متّجهَي الأوزان وأكبر قيمة في كلٍّ منهما.
# ابحث عن: "scaled dot product attention sqrt d_k variance"
# https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
# ────────────────────────────────────────────────────────────────────

D_HEAD = D_MODEL // N_HEADS
# TODO: division, softmax both, and print the scores, the weights and the two maxima.
# مهمة: softmax على الاثنين، واطبع الدرجات والأوزان وأكبر قيمتين.

### Task 2.4 — multi-head, from scratch

One attention head reads the sequence one way. Multi-head splits `d_model` into `n_heads` slices,
runs attention independently inside each slice, concatenates the results and projects them once
more. Same parameter count, several simultaneous readings.

`split_heads` and `merge_heads` are in the setup cell. **You write the attention.**

Two things to get right:

1. **`d_model % n_heads == 0`, checked with a readable error.** This is the error every student hits,
   and PyTorch's own message for it is not helpful. Raise a `ValueError` that names both numbers and
   says what the constraint is. Then trigger it on purpose and print it — a guard you have never
   seen fire is a guard you do not know works.
2. **The output shape equals the input shape.** A block that changes its own shape cannot be
   stacked, and stacking is the only reason anyone builds one.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — متعدّد الرؤوس، من الصفر

الرأس الواحد يقرأ المتتالية قراءةً واحدة. ويقسم متعدّد الرؤوس `d_model` إلى شرائح بعدد `n_heads`،
ويُشغّل الانتباه مستقلًّا داخل كل شريحة، ويضمّ النواتج ويُسقِطها مرّةً أخرى. العدد نفسه من المعاملات،
وقراءاتٌ عدّة متزامنة.

و`split_heads` و`merge_heads` في خليّة الإعداد. **وأنت تكتب الانتباه.**

وشيئان يجب إصابتهما:

١. **`d_model % n_heads == 0` مفحوصًا بخطأٍ مقروء.** فهذا الخطأ الذي يقع فيه كل طالب، ورسالة
   PyTorch له غير مُفيدة. ارفع `ValueError` يُسمّي العددين ويقول ما القيد. ثم فجّره عن قصد واطبعه —
   فالحرس الذي لم ترَه يعمل حرسٌ لا تعرف أنه يعمل.
٢. **شكل الخرج يساوي شكل الدخل.** فالكتلة التي تغيّر شكلها لا تُرصّ، والرصّ هو السبب الوحيد لبناء
   واحدة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Guard first: if d_model % n_heads, raise ValueError naming d_model, n_heads and the
#    remainder. A message a student can act on without reading your source.
# 2) Project to Q, K, V, then split_heads each one. Shapes are (1, n_heads, seq, d_head).
# 3) Scores are qh @ kh.transpose(-2, -1) / sqrt(d_head); softmax on dim=-1; then
#    weights @ vh, merge_heads, and one final projection.
# 4) Print the shape at every step. Then call it with n_heads=3 inside a try/except and
#    print the message you raised.
# Search: "multi head attention from scratch split heads pytorch"
# https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
#
# ١) الحرس أولًا: إن كان `d_model % n_heads` فارفع `ValueError` يُسمّي `d_model`
#    و`n_heads` والباقي. رسالةً يستطيع الطالب التصرّف بها بلا قراءة شيفرتك.
# ٢) أسقِط إلى Q وK وV، ثم `split_heads` كلًّا منها. والأشكال `(1, n_heads, seq, d_head)`.
# ٣) الدرجات `qh @ kh.transpose(-2, -1) / sqrt(d_head)`، وsoftmax على `dim=-1`، ثم
#    `weights @ vh` و`merge_heads` وإسقاطٌ أخير واحد.
# ٤) اطبع الشكل في كل خطوة. ثم نادِها بـ`n_heads=3` داخل `try/except` واطبع الرسالة
#    التي رفعتها.
# ابحث عن: "multi head attention from scratch split heads pytorch"
# https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
# ────────────────────────────────────────────────────────────────────

        # TODO: four projections: query, key, value and output.
        # مهمة: المُستعلِم والمفتاح والقيمة والخرج.
        # TODO: if given, softmax, mix, merge the heads and project out.
        # مهمة: وsoftmax، واخلط، واضمّ الرؤوس، وأسقِط للخرج.
# TODO: the input shape, and trigger the divisibility error on purpose to read its message.
# مهمة: وفجّر خطأ القسمة عن قصد لتقرأ رسالته.

### Task 2.5 — the whole block, with a shape after every step

The encoder block, in the order the paper puts it:

```
attention  ->  add the input back (residual)  ->  LayerNorm
           ->  feed-forward  ->  add again  ->  LayerNorm
```

Two residuals, two norms, one feed-forward that widens to `4 × d_model` and comes back. Nothing you
have not seen; the residual is week 4's skip connection and `LayerNorm` is week 3's normalisation
moved inside the model.

**Record the shape after every single step** — after the attention, after the first residual, after
the first norm, after the feed-forward, after the second residual, after the second norm — and write
the six of them to `block_shapes.json`.

Do it even though it feels like bookkeeping, because of what the last check then proves: the output
shape equals the input shape, which is what lets you write `block(block(block(x)))` and get a real
transformer. Every shape in that list being `(1, 5, 8)` is not a boring result. It is the property
the architecture is built on.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الكتلة كلها، وشكلٌ بعد كل خطوة

كتلة المُرمِّز بالترتيب الذي تضعه الورقة:

```
انتباه  ->  أضِف الدخل مرّةً أخرى (بقيّة)  ->  LayerNorm
       ->  تغذية أمامية  ->  أضِف مرّةً أخرى  ->  LayerNorm
```

بقيّتان ومعياريتان وتغذيةٌ أمامية واحدة تتوسّع إلى `4 × d_model` وتعود. ولا شيء لم ترَه؛ فالبقيّة هي
الوصلة الوثّابة في الأسبوع الرابع، و`LayerNorm` هي معيارية الأسبوع الثالث منقولةً إلى داخل النموذج.

**وسجّل الشكل بعد كل خطوة** — بعد الانتباه، وبعد البقيّة الأولى، وبعد المعيارية الأولى، وبعد
التغذية الأمامية، وبعد البقيّة الثانية، وبعد المعيارية الثانية — واكتب الستّة في
`block_shapes.json`.

افعله وإن أحسسته مسكًا للدفاتر، لما يُبرهنه الفحص الأخير بعده: شكل الخرج يساوي شكل الدخل، وهذا ما
يُتيح لك أن تكتب `block(block(block(x)))` فتحصل على محوّلٍ حقيقي. وكون كل شكل في تلك القائمة
`(1, 5, 8)` ليس نتيجةً مُملّة. بل هي الخاصّية التي بُنيت عليها البنية.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The feed-forward is Linear(d_model, d_ff) -> activation -> Linear(d_ff, d_model).
#    nn.Sequential holds all three.
# 2) A residual is literally x + sublayer(x). If the shapes did not match you could not
#    write the plus sign — which is why the shape list is the proof, not a formality.
# 3) Record each intermediate shape into a dict as you go: tuple(t.shape).
# 4) Return the output and the recorded shapes so the artefact cell can write them out.
# Search: "transformer encoder block residual layernorm feedforward order"
# https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
#
# ١) التغذية الأمامية هي `Linear(d_model, d_ff)` ثم تنشيط ثم `Linear(d_ff, d_model)`.
#    و`nn.Sequential` تحمل الثلاثة.
# ٢) والبقيّة حرفيًا `x + sublayer(x)`. فلو لم تتحاذَ الأشكال لما استطعت كتابة علامة
#    الجمع — ولهذا تكون قائمة الأشكال هي البرهان لا إجراءً شكليًا.
# ٣) سجّل كل شكل وسيط في قاموس أثناء المسير: `tuple(t.shape)`.
# ٤) أعِد الخرج والأشكال المُسجَّلة لتكتبها خليّة الأثر.
# ابحث عن: "transformer encoder block residual layernorm feedforward order"
# https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
# ────────────────────────────────────────────────────────────────────

        # TODO: Build the attention, the two LayerNorms and the feed-forward stack.
        # مهمة: ابنِ الانتباه، ومعياريّتَي الطبقة، ومكدّس التغذية الأمامية.
        # TODO: (output, weights, shapes).
        # مهمة: `(output, weights, shapes)`.
# TODO: input shape, and confirm stacking the block three times still works.
# مهمة: رصّ الكتلة ثلاث مرّات ما زال يعمل.

### Task 2.6 — against PyTorch, then against a real review

Two checks, and both are the kind you should run on anything you reimplement.

**Against the library.** Build an `nn.MultiheadAttention` with `batch_first=True`, copy your
projection weights into it — its `in_proj_weight` is the query, key and value weights stacked into
one `(3·d_model, d_model)` tensor, in that order — and compare the attention weights against yours.
They must agree to **1e-5**. If they do not, the usual causes are the stacking order, the missing
bias, or `average_attn_weights` quietly averaging your heads away.

**Against real input.** Take one review, tokenise it on whitespace against a vocabulary built from
the corpus, embed the ids with `nn.Embedding`, add the positional encoding from task 2.1, and push
the sequence through the block. No training and no label — the only question is whether a real
variable-length sequence comes out with the shape it went in with.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — مقابل PyTorch، ثم مقابل مراجعة حقيقية

فحصان، وكلاهما من النوع الذي يجدر إجراؤه على كل ما تُعيد تنفيذه.

**مقابل المكتبة.** ابنِ `nn.MultiheadAttention` بـ`batch_first=True`، وانسخ أوزان إسقاطك إليها —
فـ`in_proj_weight` عندها أوزان المُستعلِم والمفتاح والقيمة مرصوصةً في مُوتِّرٍ واحد
`(3·d_model, d_model)` بذلك الترتيب — وقارن أوزان الانتباه بأوزانك. ويجب أن تتّفقا إلى **١e−٥**.
فإن لم تتّفقا فالأسباب المعتادة ترتيب الرصّ، أو الانحياز الغائب، أو `average_attn_weights` يُوسِّط
رؤوسك بهدوء فتذهب.

**ومقابل دخلٍ حقيقي.** خُذ مراجعةً واحدة، وقسّمها على المسافات مقابل معجمٍ مبنيّ من المُدوّنة، ومثّل
المعرّفات بـ`nn.Embedding`، وأضِف ترميز الموضع من المهمة ٢٫١، وادفع المتتالية عبر الكتلة. ولا تدريب
ولا تسمية — والسؤال الوحيد هل تخرج متتالية حقيقية متغيّرة الطول بالشكل الذي دخلت به.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) in_proj_weight is (3 * d_model, d_model): rows 0:d are the query, d:2d the key,
#    2d:3d the value. in_proj_bias is the same three biases concatenated.
# 2) Call the reference with need_weights=True AND average_attn_weights=False, or it
#    returns the mean over heads and your (1, n_heads, seq, seq) tensor will not compare.
# 3) Wrap the copies in torch.no_grad() and compare with .abs().max().
# 4) For the review: re.findall(r"[a-z']+", text.lower()), map to ids through a dict
#    built from the corpus, then nn.Embedding(len(vocab), D_MODEL).
# Search: "nn.MultiheadAttention in_proj_weight layout average_attn_weights"
# https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
#
# ١) و`in_proj_weight` بشكل `(3 * d_model, d_model)`: الصفوف `0:d` للمُستعلِم و`d:2d`
#    للمفتاح و`2d:3d` للقيمة. و`in_proj_bias` هي الانحيازات الثلاثة نفسها مضمومة.
# ٢) نادِ المرجع بـ`need_weights=True` **و**`average_attn_weights=False`، وإلّا أعاد
#    المتوسط على الرؤوس ولن يُقارن مُوتِّرك `(1, n_heads, seq, seq)`.
# ٣) لُفّ النسخ في `torch.no_grad()` وقارن بـ`.abs().max()`.
# ٤) وللمراجعة: `re.findall(r"[a-z']+", text.lower())`، وحوّلها معرّفات بقاموسٍ مبنيّ من
#    المُدوّنة، ثم `nn.Embedding(len(vocab), D_MODEL)`.
# ابحث عن: "nn.MultiheadAttention in_proj_weight layout average_attn_weights"
# https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
# ────────────────────────────────────────────────────────────────────

# TODO: report the largest difference in the attention weights and in the outputs.
# مهمة: واعرض أكبر فرق في أوزان الانتباه وفي الخرجين.
# TODO: encoding, run it through the block, and confirm the shape survived.
# مهمة: عبر الكتلة، وتأكّد أن الشكل نجا.

## Section 3 — Stretch: the causal mask  (≈30 min)

The block you built lets every token attend to every other token, including the ones after it. For
an encoder classifying a finished review, that is correct — the whole text is available.

For a model that **generates** text, it is fatal. Predicting token 5 while being allowed to look at
token 6 is not prediction; it is reading the answer.

The fix is one line: set the upper triangle of the score matrix to `-inf` before the softmax, so
`exp` sends those weights to exactly zero. Build the mask with `torch.triu`, pass it through the
`mask` argument your `MultiHeadAttention.forward` already accepts, and print the resulting weight
matrix. It must be lower-triangular, and every row must still sum to 1 — including row 0, which now
attends to exactly one token, itself.

Then write the two sentences. The question is not "what does the mask do" — you just printed that.
It is: **why does training without it produce a beautiful loss curve and a useless model?**

<div dir="rtl" align="right">

## القسم الثالث — التمديد: القناع السببي (نحو ٣٠ دقيقة)

الكتلة التي بنيتها تُتيح لكل رمز أن ينتبه إلى كل رمزٍ آخر، ومنها ما بعده. وهذا صحيح لمُرمِّزٍ يُصنّف
مراجعةً منتهية — فالنص كله متاح.

وهو قاتل لنموذجٍ **يُولّد** نصًّا. فالتنبّؤ بالرمز الخامس والنظر إلى السادس مباحٌ ليس تنبّؤًا بل
قراءةً للجواب.

والإصلاح سطر واحد: اجعل المثلّث الأعلى من مصفوفة الدرجات `-inf` قبل softmax، فيُرسل `exp` تلك
الأوزان إلى الصفر تمامًا. ابنِ القناع بـ`torch.triu`، ومرّره عبر وسيط `mask` الذي تقبله
`MultiHeadAttention.forward` عندك أصلًا، واطبع مصفوفة الأوزان الناتجة. ويجب أن تكون مثلّثيةً سفلية،
وأن يجمع كل صفٍّ إلى واحد — ومنها الصف صفر، الذي ينتبه الآن إلى رمزٍ واحد بالضبط، نفسه.

ثم اكتب الجملتين. والسؤال ليس «ما يفعله القناع» — فقد طبعته الآن. بل: **لماذا يُنتج التدريب بدونه
منحنى خسارةٍ جميلًا ونموذجًا عديم النفع؟**

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) torch.triu(torch.ones(seq, seq, dtype=torch.bool), diagonal=1) is True exactly
#    where a token would be looking ahead.
# 2) The mask broadcasts over (batch, n_heads, seq, seq) on its own — no need to expand.
# 3) Check lower-triangularity by asserting the masked positions are exactly 0.0, not
#    just small. -inf through a softmax is an exact zero.
# 4) Plot both weight matrices side by side and print the row sums.
# Search: "causal mask torch.triu masked_fill attention"
# https://pytorch.org/docs/stable/generated/torch.triu.html
#
# ١) `torch.triu(torch.ones(seq, seq, dtype=torch.bool), diagonal=1)` صحيحة بالضبط حيث
#    ينظر الرمز إلى الأمام.
# ٢) والقناع يُبثّ على `(batch, n_heads, seq, seq)` من نفسه — فلا حاجة للتوسيع.
# ٣) افحص المثلّثية السفلية بالتأكّد أن المواضع المُقنَّعة صفر تمامًا لا صغيرة فقط. فـ
#    `-inf` عبر softmax صفرٌ تامّ.
# ٤) ارسم مصفوفتَي الأوزان جنبًا إلى جنب واطبع مجاميع الصفوف.
# ابحث عن: "causal mask torch.triu masked_fill attention"
# https://pytorch.org/docs/stable/generated/torch.triu.html
# ────────────────────────────────────────────────────────────────────

# TODO: WHY_THE_MASK_MATTERS.
# مهمة: تجمع إلى واحد، وارسم المُقنَّعة مقابل غير المُقنَّعة، واكتب `WHY_THE_MASK_MATTERS`.

## Save your artefact

`block_shapes.json` — the shape after every step of the block, the head dimension, the parameter
count, the agreement with `nn.MultiheadAttention`, and the causal-mask result.

It is a small file and it does a specific job: it is the record that your block is
**shape-preserving**, which is the property that lets it be stacked. Tomorrow you stop building
blocks and start using six of them at once, pretrained, and the reason you can trust that stack is
that you verified one.

<div dir="rtl" align="right">

## احفظ أثرك

`block_shapes.json` — الشكل بعد كل خطوة من الكتلة، وبُعد الرأس، وعدد المعاملات، والاتّفاق مع
`nn.MultiheadAttention`، ونتيجة القناع السببي.

وهو ملف صغير يقوم بعملٍ محدّد: فهو سجلّ أن كتلتك **حافظة للشكل**، وهي الخاصّية التي تُتيح رصّها.
وغدًا تتوقّف عن بناء الكتل وتبدأ باستخدام ستٍّ منها معًا، مُدرَّبةً مسبقًا، وسبب ثقتك بذلك الرصّ أنك
تحقّقت من واحدة.

</div>


In [ ]:
block_shapes = {
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "d_head": D_HEAD,
    "d_ff": D_FF,
    "parameters": PARAMETER_COUNT,
    "shapes": {step: list(shape) for step, shape in BLOCK_SHAPES.items()},
    "shape_preserving": bool(BLOCK_PRESERVES_SHAPE),
    "real_review_shapes": {step: list(shape) for step, shape in REAL_SHAPES.items()},
    "matches_torch_multihead": {
        "max_weight_difference": WEIGHT_DIFFERENCE,
        "max_output_difference": OUTPUT_DIFFERENCE,
        "within_1e-5": bool(WEIGHT_DIFFERENCE < 1e-5),
    },
    "positional_encoding": {"positions_differ": bool(POSITIONS_DIFFER),
                            "stable_across_calls": bool(POSITION_IS_STABLE)},
    "causal_mask": {"lower_triangular": IS_LOWER_TRIANGULAR,
                    "rows_sum_to_one": MASKED_ROWS_SUM_TO_ONE},
    "warm_up": {"order_invisible_without_positions": bool(ORDER_INVISIBLE_WITHOUT),
                "order_visible_with_positions": bool(ORDER_VISIBLE_WITH)},
}

SHAPES_PATH = ARTEFACT_DIR / "block_shapes.json"
SHAPES_PATH.write_text(json.dumps(block_shapes, indent=2), encoding="utf-8")

print(json.dumps({k: v for k, v in block_shapes.items()
                  if k in {"shapes", "shape_preserving", "parameters",
                           "matches_torch_multihead"}}, indent=2))
print(f"\nwrote {SHAPES_PATH.name} — {len(BLOCK_SHAPES)} steps recorded")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>


In [ ]:
check(ORDER_INVISIBLE_WITHOUT and ORDER_VISIBLE_WITH,
      f"every reordering must give the identical output without positions, and at least one must "
      f"change it once positions are added — invisible everywhere: {ORDER_INVISIBLE_WITHOUT}, "
      f"visible somewhere: {ORDER_VISIBLE_WITH}. This is the whole warm-up: one addition is what "
      f"makes order exist — and the full reversal alone would not have shown it",
      f"يجب أن يُعطي كل إعادة ترتيب الخرجَ المتطابق بلا مواضع، وأن يُغيّره واحدٌ منها على الأقل بعد "
      f"إضافة المواضع — غير مرئي في الكل: {ORDER_INVISIBLE_WITHOUT}، ومرئي في واحد: "
      f"{ORDER_VISIBLE_WITH}. وهذا هو الإحماء كله: إضافةٌ واحدة هي ما يُوجد الترتيب — والعكس "
      f"الكامل وحده لم يكن ليُظهره")

check(POSITIONS_DIFFER and POSITION_IS_STABLE,
      f"two positions must get different encodings and the same position must be identical across "
      f"calls — differ: {POSITIONS_DIFFER}, stable: {POSITION_IS_STABLE}. A failure on the first "
      f"is an exponent bug; on the second, you learned it instead of computing it",
      f"يجب أن يأخذ موضعان ترميزين مختلفين وأن يكون الموضع نفسه متطابقًا بين النداءات — يختلفان: "
      f"{POSITIONS_DIFFER}، ومستقرّ: {POSITION_IS_STABLE}. والفشل في الأول خلل أُسّ، وفي الثاني "
      f"أنك تعلّمته بدل أن تحسبه")

check(PROJECTION_MATCHES and SCALED_IS_SMALLER,
      f"the hand-computed projection must match nn.Linear and the scaled score must be smaller "
      f"than the unscaled one — projection matches: {PROJECTION_MATCHES}, scaled is smaller: "
      f"{SCALED_IS_SMALLER}",
      f"يجب أن يُطابق الإسقاط المحسوب يدويًا `nn.Linear` وأن تكون الدرجة المقيسة أصغر من غير "
      f"المقيسة — الإسقاط مطابق: {PROJECTION_MATCHES}، والمقيسة أصغر: {SCALED_IS_SMALLER}")

check(DIVISIBILITY_GUARD_FIRED,
      f"MultiHeadAttention(d_model={D_MODEL}, n_heads=3) must raise your own ValueError — a guard "
      f"you have never seen fire is a guard you do not know works",
      f"يجب أن ترفع `MultiHeadAttention(d_model={D_MODEL}, n_heads=3)` خطأ `ValueError` الخاصّ بك "
      f"— فالحرس الذي لم ترَه يعمل حرسٌ لا تعرف أنه يعمل")

check_shape(block_output, (1, SEQ_LEN, D_MODEL),
            "the block's output must have exactly the input's shape, or it cannot be stacked",
            "يجب أن يكون لخرج الكتلة شكل الدخل بالضبط، وإلّا لم تُمكن رصّها")

check(BLOCK_PRESERVES_SHAPE and REAL_SEQUENCE_SURVIVED
      and all(tuple(s) == tuple(BLOCK_SHAPES["input"]) for s in BLOCK_SHAPES.values()),
      f"every recorded step must keep the same shape, and a real {ids.shape[1]}-token review must "
      f"pass through a block sized for {SEQ_LEN} — shapes: {list(BLOCK_SHAPES.values())}, real "
      f"review survived: {REAL_SEQUENCE_SURVIVED}",
      f"يجب أن يُبقي كل خطوة مُسجَّلة الشكل نفسه، وأن تعبر مراجعةٌ حقيقية بـ{ids.shape[1]} رمزًا "
      f"كتلةً مقيسة لـ{SEQ_LEN} — الأشكال: {list(BLOCK_SHAPES.values())}، والمراجعة نجت: "
      f"{REAL_SEQUENCE_SURVIVED}")

check(WEIGHT_DIFFERENCE < 1e-5,
      f"your multi-head attention weights must match nn.MultiheadAttention to 1e-5 — largest "
      f"difference {WEIGHT_DIFFERENCE:.2e}. The usual causes are the in_proj stacking order, a "
      f"forgotten bias, or average_attn_weights averaging your heads away",
      f"يجب أن تُطابق أوزان انتباهك متعدّد الرؤوس `nn.MultiheadAttention` إلى ١e−٥ — وأكبر فرق "
      f"{WEIGHT_DIFFERENCE:.2e}. والأسباب المعتادة ترتيب رصّ `in_proj`، أو انحياز منسيّ، أو "
      f"`average_attn_weights` يُوسِّط رؤوسك فتذهب")

check(IS_LOWER_TRIANGULAR and MASKED_ROWS_SUM_TO_ONE,
      f"the causal-masked weight matrix must be strictly lower-triangular with every row still "
      f"summing to 1 — lower-triangular: {IS_LOWER_TRIANGULAR}, rows sum to 1: "
      f"{MASKED_ROWS_SUM_TO_ONE}. Weights above the diagonal must be exactly 0, not small",
      f"يجب أن تكون مصفوفة الأوزان المُقنَّعة سببيًا مثلّثيةً سفليةً تمامًا وأن يجمع كل صف إلى واحد "
      f"— مثلّثية سفلية: {IS_LOWER_TRIANGULAR}، والصفوف تجمع إلى واحد: {MASKED_ROWS_SUM_TO_ONE}. "
      f"ويجب أن تكون الأوزان فوق القُطر صفرًا تمامًا لا صغيرة")

report()

## What's next

**W5D5 — Fine-tuning BERT.** You have built one encoder block with 8 dimensions and 2 heads.
`distilbert-base-uncased` is six of them with 768 dimensions and 12 heads, already trained on
several billion words, and tomorrow you download it and adapt it to Monday's 12,000 reviews in one
epoch.

The block is not a metaphor for what is inside it. It is what is inside it — with a real subword
tokeniser in front, a classification head on top, and the shape discipline you practised today doing
the work when a tensor comes out `(16, 128, 768)` instead of `(16, 768)`.

Tomorrow also settles the week: same data, same folds, TF-IDF against BERT, with accuracy, F1, fit
time, inference time and disk size in one table — and a recommendation you have to defend.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٥ — الضبط الدقيق لـBERT.** بنيت كتلة مُرمِّزٍ واحدة بثمانية أبعاد ورأسين.
و`distilbert-base-uncased` ستٌّ منها بـ٧٦٨ بُعدًا واثني عشر رأسًا، مُدرَّبةً أصلًا على عدّة مليارات
كلمة، وغدًا تُنزّلها وتُكيّفها على مراجعات الاثنين الاثنتي عشرة ألفًا في حقبةٍ واحدة.

والكتلة ليست استعارةً لما فيها. بل هي ما فيها — ومعها مُقسِّم جزئي حقيقي أمامها، ورأس تصنيفٍ فوقها،
وانضباط الأشكال الذي تمرّنت عليه اليوم يقوم بالعمل حين يخرج مُوتِّر `(16, 128, 768)` بدل
`(16, 768)`.

والغد يحسم الأسبوع أيضًا: البيانات نفسها والأثلام نفسها، وTF-IDF مقابل BERT، بالدقّة ومقياس F1 وزمن
التدريب وزمن الاستدلال وحجم القرص في جدولٍ واحد — وتوصيةٍ عليك الدفاع عنها.

</div>
